# 1. Load Data and Inspect Columns
Read comment and weibo CSVs, print shape, list columns, and preview a sample row.

In [1]:
import pandas as pd
import numpy as np
import os

RESULTS_DIR = 'D:/010_CodePrograms/L/LLM_su7/results'
ROOT_DIR = 'D:/010_CodePrograms/L/LLM_su7'

print("【加载数据】")
comments_df = pd.read_csv(f"{RESULTS_DIR}/热门微博评论数据_去重.csv", encoding='utf-8-sig')
weibo_df = pd.read_csv(f"{RESULTS_DIR}/热门微博_去重.csv", encoding='utf-8-sig')

print(f"comments_df: {comments_df.shape}")
print(f"weibo_df: {weibo_df.shape}")
print("\ncomments_df columns:")
print(list(comments_df.columns))
print("\nweibo_df columns:")
print(list(weibo_df.columns))

print("\ncomments_df sample row:")
display(comments_df.head(1))
print("\nweibo_df sample row:")
display(weibo_df.head(1))

【加载数据】
comments_df: (271452, 18)
weibo_df: (4186, 10)

comments_df columns:
['原文链接', '根评论ID', '父评论ID', '评论ID', '用户ID', '父用户昵称', '用户昵称', '评论内容', '发布时间', '子评论数', '点赞数', '用户认证', '用户总评论数', '用户总转发数', '用户总点赞数', '是否是一级评论', 'crawl_date', '内容清洗']

weibo_df columns:
['author_name', 'author_url', 'weibo_url', 'publish_time', 'weibo_content', 'repost_count', 'comment_count', 'like_count', 'crawl_date', '内容清洗']

comments_df sample row:


,原文链接,根评论ID,父评论ID,评论ID,用户ID,父用户昵称,用户昵称,评论内容,发布时间,子评论数,点赞数,用户认证,用户总评论数,用户总转发数,用户总点赞数,是否是一级评论,crawl_date,内容清洗
0,https://weibo.com/7871239944/PkvsRdOvf,NaN,7871239944,5148876097454178,6865028041,NaN,挖土的小铜钱,非常好非常好，好产品应当像小米汽车一样会自动宣传[鼓掌][鼓掌][鼓掌][打call][打c...,2025-03-27 18:26:25,3,127.0,True,25928,610,40859,True,2025-03-27,非常好非常好，好产品应当像小米汽车一样会自动宣传[鼓掌][鼓掌][鼓掌][打call][打c...



weibo_df sample row:


,author_name,author_url,weibo_url,publish_time,weibo_content,repost_count,comment_count,like_count,crawl_date,内容清洗
0,小米汽车,https://weibo.com/7871239944,https://weibo.com/7871239944/PkvsRdOvf,2025-03-27 18:22:00,众所周知，小米汽车的广告都是车主拍的👏 一起来欣赏大片，小米SU7 Ultra车主出品。 L...,48,105,2017.0,2025-03-27,众所周知，小米汽车的广告都是车主拍的👏 一起来欣赏大片，小米SU7 Ultra车主出品。 L...


# 2. Normalize Column Names and Types
Standardize column names, normalize ID fields, parse dates, and clean numeric fields.

In [2]:
# 标准化列名（去除首尾空格）
comments_df.columns = comments_df.columns.str.strip()
weibo_df.columns = weibo_df.columns.str.strip()

# 兼容列名（用户认证/是否一级评论）
if '用户是否认证' in comments_df.columns and '用户认证' not in comments_df.columns:
    comments_df = comments_df.rename(columns={'用户是否认证': '用户认证'})
if '是否一级评论' in comments_df.columns and '是否是一级评论' not in comments_df.columns:
    comments_df = comments_df.rename(columns={'是否一级评论': '是否是一级评论'})

# 标准化评论ID（移除.0后缀）
def normalize_id(id_val):
    if pd.isna(id_val):
        return None
    try:
        return str(int(float(id_val)))
    except Exception:
        return str(id_val)

id_cols = ['评论ID', '根评论ID', '父评论ID']
for col in id_cols:
    if col in comments_df.columns:
        comments_df[f'{col}_str'] = comments_df[col].apply(normalize_id)

# 解析发布时间
if '发布时间' in comments_df.columns:
    comments_df['发布时间'] = pd.to_datetime(comments_df['发布时间'], errors='coerce')

# 数值清洗：处理带逗号的数字
def clean_number(x):
    if isinstance(x, str):
        x = x.replace(',', '')
    try:
        return int(float(x))
    except Exception:
        return 0

numeric_cols = [
    '用户总转发数', '用户总评论数', '用户总点赞数',
    '子评论数', '点赞数',
    '微博转发数', '微博评论数', '微博点赞数',
    'repost_count', 'comment_count', 'like_count'
 ]
for col in numeric_cols:
    if col in comments_df.columns:
        comments_df[col] = comments_df[col].apply(clean_number)
for col in ['repost_count', 'comment_count', 'like_count']:
    if col in weibo_df.columns:
        weibo_df[col] = weibo_df[col].apply(clean_number)

print("完成列名与类型标准化")

完成列名与类型标准化


# 3. Validate Required Feature Columns
Define required columns, compare against the actual schema, and report missing/extra columns.

In [3]:
required_comment_cols = [
    '评论内容', '原文链接', '发布时间',
    '用户总转发数', '用户总评论数', '用户总点赞数',
    '用户认证', '是否是一级评论',
    '子评论数', '点赞数',
    '评论ID_str', '根评论ID_str', '父评论ID_str'
 ]

missing = [c for c in required_comment_cols if c not in comments_df.columns]
extra = [c for c in comments_df.columns if c not in required_comment_cols]

print("缺失列:", missing)
print("额外列(展示部分):", extra[:10])

if missing:
    raise ValueError(f"缺失必要列: {missing}")

缺失列: []
额外列(展示部分): ['根评论ID', '父评论ID', '评论ID', '用户ID', '父用户昵称', '用户昵称', 'crawl_date', '内容清洗']


# 4. Rebuild Feature Table Safely
Create derived columns, then select only existing columns and rename to the final schema.

In [4]:
# 微博特征映射（去重）
weibo_dedup = weibo_df.drop_duplicates(subset=['weibo_url'], keep='first')
weibo_features = weibo_dedup.set_index('weibo_url')[['weibo_content', 'repost_count', 'comment_count', 'like_count']]
weibo_features.columns = ['微博文案', '微博转发数', '微博评论数', '微博点赞数']

# 评论ID -> 评论内容映射
comment_content_map = dict(zip(comments_df['评论ID_str'], comments_df['评论内容']))

# 合并微博特征
comments_df = comments_df.merge(
    weibo_features.reset_index(),
    left_on='原文链接',
    right_on='weibo_url',
    how='left'
).drop(columns=['weibo_url'])

# 根评论/父评论文案
comments_df['根评论文案'] = comments_df['根评论ID_str'].map(comment_content_map)
def get_parent_content(row):
    if row['是否是一级评论'] == 1:
        return row['微博文案']
    return comment_content_map.get(row['父评论ID_str'], None)
comments_df['父评论文案'] = comments_df.apply(get_parent_content, axis=1)

# 过滤点赞数缺失
before_rows = len(comments_df)
comments_df = comments_df[comments_df['点赞数'].notna()].copy()
print(f"过滤点赞数缺失: {before_rows - len(comments_df):,} 条")

# 计算调整后的用户总评论数/点赞数（与0取max）
comments_df['用户总评论数_adj'] = (comments_df['用户总评论数'] - comments_df['子评论数']).clip(lower=0)
comments_df['用户总点赞数_adj'] = (comments_df['用户总点赞数'] - comments_df['点赞数']).clip(lower=0)

# 选择特征列（安全选择）
feature_cols = [
    '评论内容', '微博文案', '根评论文案', '父评论文案', '发布时间',
    '用户总转发数', '用户总评论数_adj', '用户总点赞数_adj',
    '用户认证', '是否是一级评论', '子评论数', '点赞数',
    '微博转发数', '微博评论数', '微博点赞数'
 ]
missing_feature_cols = [c for c in feature_cols if c not in comments_df.columns]
if missing_feature_cols:
    raise ValueError(f"缺失特征列: {missing_feature_cols}")

feature_df = comments_df[feature_cols].copy()

# 重命名列
feature_df.columns = [
    '评论文案', '微博文案', '根评论文案', '父评论文案', '发布时间',
    '用户总转发数', '用户总评论数', '用户总点赞数',
    '用户是否认证', '是否一级评论', '子评论数', '点赞数',
    '微博转发数', '微博评论数', '微博点赞数'
 ]

print(f"特征表大小: {len(feature_df):,} 条")
print(f"特征列 ({len(feature_df.columns)}列): {list(feature_df.columns)}")

过滤点赞数缺失: 0 条
特征表大小: 271,452 条
特征列 (15列): ['评论文案', '微博文案', '根评论文案', '父评论文案', '发布时间', '用户总转发数', '用户总评论数', '用户总点赞数', '用户是否认证', '是否一级评论', '子评论数', '点赞数', '微博转发数', '微博评论数', '微博点赞数']


# 5. Split Train/Val/Test with Rules
Shuffle and split into 8:1:1, then verify sizes and overlaps.

In [5]:
np.random.seed(42)
feature_df = feature_df.sample(frac=1, random_state=42).reset_index(drop=True)

total = len(feature_df)
val_size = int(total * 0.1)
test_size = int(total * 0.1)
train_size = total - val_size - test_size

train_df = feature_df.iloc[:train_size].copy()
val_df = feature_df.iloc[train_size:train_size + val_size].copy()
test_df = feature_df.iloc[train_size + val_size:].copy()

print(f"训练集: {len(train_df):,} ({len(train_df)/total*100:.1f}%)")
print(f"验证集: {len(val_df):,} ({len(val_df)/total*100:.1f}%)")
print(f"测试集: {len(test_df):,} ({len(test_df)/total*100:.1f}%)")
print(f"总计: {len(train_df)+len(val_df)+len(test_df):,} = {total:,}")

训练集: 217,162 (80.0%)
验证集: 27,145 (10.0%)
测试集: 27,145 (10.0%)
总计: 271,452 = 271,452


# 6. Save Outputs and Verify
Save CSV/PKL, print file sizes, and reload a saved file to validate schema and row counts.

In [6]:
# 添加全局唯一序号
train_df.insert(0, '序号', range(0, len(train_df)))
val_start = len(train_df)
val_df.insert(0, '序号', range(val_start, val_start + len(val_df)))
test_start = val_start + len(val_df)
test_df.insert(0, '序号', range(test_start, test_start + len(test_df)))

# 保存CSV
train_df.to_csv(f'{ROOT_DIR}/train.csv', index=False, encoding='utf-8-sig')
val_df.to_csv(f'{ROOT_DIR}/val.csv', index=False, encoding='utf-8-sig')
test_df.to_csv(f'{ROOT_DIR}/test.csv', index=False, encoding='utf-8-sig')

# 保存PKL
train_df.to_pickle(f'{ROOT_DIR}/train.pkl')
val_df.to_pickle(f'{ROOT_DIR}/val.pkl')
test_df.to_pickle(f'{ROOT_DIR}/test.pkl')

print("保存完成")
for name in ['train', 'val', 'test']:
    csv_size = os.path.getsize(f'{ROOT_DIR}/{name}.csv') / 1024 / 1024
    pkl_size = os.path.getsize(f'{ROOT_DIR}/{name}.pkl') / 1024 / 1024
    print(f"{name}: CSV={csv_size:.1f}MB, PKL={pkl_size:.1f}MB")

# 重新加载验证
train_check = pd.read_pickle(f'{ROOT_DIR}/train.pkl')
val_check = pd.read_pickle(f'{ROOT_DIR}/val.pkl')
test_check = pd.read_pickle(f'{ROOT_DIR}/test.pkl')

print("\n校验: 行数")
print(len(train_check), len(val_check), len(test_check))
print("\n校验: 列")
print(list(train_check.columns))
print("\n序号范围:")
print(train_check['序号'].min(), train_check['序号'].max())
print(val_check['序号'].min(), val_check['序号'].max())
print(test_check['序号'].min(), test_check['序号'].max())

保存完成
train: CSV=221.0MB, PKL=38.1MB
val: CSV=27.5MB, PKL=6.9MB
test: CSV=27.9MB, PKL=6.8MB

校验: 行数
217162 27145 27145

校验: 列
['序号', '评论文案', '微博文案', '根评论文案', '父评论文案', '发布时间', '用户总转发数', '用户总评论数', '用户总点赞数', '用户是否认证', '是否一级评论', '子评论数', '点赞数', '微博转发数', '微博评论数', '微博点赞数']

序号范围:
0 217161
217162 244306
244307 271451
